# 01 — Data Gathering (Live NOAA NDBC) → S3 (Curated)

This notebook **downloads live NOAA NDBC buoy data** (no backup files) and writes:

- **Parquet** to S3 for Athena / analytics
- **CSV** to S3 for simple pipeline ingestion

It also writes a small **manifest** so downstream notebooks know what was produced.


In [ ]:
# Install deps (run once per kernel)
%pip install -q -r ../requirements.txt

In [ ]:
import sys
from pathlib import Path

# Make local package imports work
repo_root = Path.cwd().parent  # notebooks/ -> repo root
sys.path.insert(0, str(repo_root))

import json
import time
import boto3
import sagemaker
import pandas as pd

from src.data_utils import fetch_ndbc_data, clean_ndbc_data
from src.s3_utils import get_s3_client, get_sagemaker_bucket, verify_bucket, upload_df_to_s3

In [ ]:
# AWS context
sess = sagemaker.Session()
region = boto3.Session().region_name
bucket = get_sagemaker_bucket(sess)
s3_client = get_s3_client(region_name=region)

print("Region:", region)
print("Default bucket:", bucket)

verify_bucket(bucket, s3_client=s3_client)

In [ ]:
# ---- CONFIG (edit these for your project) ----
BUOY_IDS = [
    "46086",
    "46042",
    "46011",
]

START_DATE = "2023-01-01"
END_DATE   = "2026-02-01"

CURATED_PREFIX = "curated/ndbc"  # output prefix in S3

# File names inside each buoy=XXXX partition folder
PARQUET_NAME = "stdmet.parquet"
CSV_NAME     = "stdmet.csv"
# ----------------------------------------------

In [ ]:
# Download -> clean -> upload (per buoy)
results = {}
row_counts = {}

for buoy_id in BUOY_IDS:
    print("\n" + "="*80)
    print("Buoy:", buoy_id)

    try:
        raw_df = fetch_ndbc_data(buoy_id, START_DATE, END_DATE, mode="stdmet")
    except Exception as e:
        print(f"[WARN] Fetch failed for buoy {buoy_id}: {e}")
        continue

    if raw_df is None or len(raw_df) == 0:
        print(f"[WARN] No data returned for buoy {buoy_id}. Skipping.")
        continue

    clean_df = clean_ndbc_data(raw_df, buoy_id)

    print("Rows (raw):", len(raw_df))
    print("Rows (clean):", len(clean_df))
    row_counts[buoy_id] = int(len(clean_df))

    # Upload Parquet + CSV
    parquet_uri = upload_df_to_s3(
        clean_df, bucket, CURATED_PREFIX, buoy_id,
        file_name=PARQUET_NAME,
        file_format="parquet",
        s3_client=s3_client,
    )
    csv_uri = upload_df_to_s3(
        clean_df, bucket, CURATED_PREFIX, buoy_id,
        file_name=CSV_NAME,
        file_format="csv",
        s3_client=s3_client,
    )

    results[buoy_id] = {"parquet": parquet_uri, "csv": csv_uri}

print("\nDONE. Uploaded buoys:", list(results.keys()))
results

In [ ]:
# Quick dataset sanity checks (AAI-540 rule of thumb: 3-5 files, 2 files >= 10k rows)
print("Row counts per buoy:")
display(pd.Series(row_counts, name="rows").sort_values(ascending=False))

num_files = len(results)
num_10k = sum(1 for v in row_counts.values() if v >= 10_000)
print("\nFiles uploaded:", num_files)
print("Files with >=10k rows:", num_10k)

if num_files < 3:
    raise RuntimeError("Need at least 3 buoy files uploaded to meet dataset size guidance. Adjust BUOY_IDS.")
if num_10k < 2:
    raise RuntimeError("Need at least 2 buoy files with >=10k rows. Expand date range or choose different buoys.")

In [ ]:
# Write a manifest (local + S3)
manifest = {
    "created_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
    "bucket": bucket,
    "region": region,
    "curated_prefix": CURATED_PREFIX,
    "start_date": START_DATE,
    "end_date": END_DATE,
    "buoy_ids": list(results.keys()),
    "row_counts": row_counts,
    "artifacts": results,
}

manifest_path = repo_root / "manifests"
manifest_path.mkdir(exist_ok=True)
local_manifest = manifest_path / "curated_manifest.json"
local_manifest.write_text(json.dumps(manifest, indent=2), encoding="utf-8")

# Upload manifest too
manifest_s3_uri = upload_df_to_s3(
    pd.DataFrame([manifest]),
    bucket=bucket,
    s3_prefix="manifests",
    buoy_id="all",
    file_name="curated_manifest.csv",
    file_format="csv",
    s3_client=s3_client,
)

print("Local manifest:", local_manifest)
print("S3 manifest CSV:", manifest_s3_uri)

In [ ]:
# Store variables for downstream notebooks
%store bucket
%store region
%store CURATED_PREFIX
%store BUOY_IDS
%store manifest_s3_uri